## Carga


Para el proceso de carga hemos definido un esquema estrella que nos permite
organizar los datos de manera eficiente para consultas analíticas. El esquema
estrella consta de una tabla central de hechos y varias tablas de dimensiones
que la rodean. Todos estos datos serán almacenados en una base de datos
relacional, como PostgreSQL, para facilitar el acceso y la gestión de los datos.
Se tiene el siguiente esquema:

```mermaid
erDiagram
    DIM_TIME {
        int id PK
        date date
        time time
        int year
        int month
        int day
        string day_of_week
        string turn
    }

    DIM_LOCATION {
        int id PK
        int zone_number
        string sector_name
        string decentralized_base
    }

    DIM_CASE_TYPE {
        int id PK
        string category
    }

    DIM_ORIGIN {
        int id PK
        string category
    }

    FACT_INCIDENT {
        int id PK
        int id_time FK
        int id_location FK
        int id_case_type FK
        int id_origin FK
        string case_number
        float latitude
        float longitude
        float response_time_min
        date attention_date
    }

    DIM_TIME ||--o{ FACT_INCIDENT : "has"
    DIM_LOCATION ||--o{ FACT_INCIDENT : "has"
    DIM_CASE_TYPE ||--o{ FACT_INCIDENT : "has"
    DIM_ORIGIN ||--o{ FACT_INCIDENT : "has"
```


In [1]:
from datetime import datetime
from decimal import Decimal

import pandas as pd
from IPython.display import Markdown
from tortoise.transactions import in_transaction

from registro_de_indicencias.database import close_db, init_db, migrate_db
from registro_de_indicencias.folder import CLEAN_CSV_PATH
from registro_de_indicencias.model import (
    DimCaseType,
    DimLocation,
    DimOrigin,
    DimTime,
    FactIncident,
)

In [2]:
await migrate_db()

In [3]:
df = pd.read_csv(
    CLEAN_CSV_PATH,
    encoding="utf-8",
    parse_dates=["case_date", "case_attention"],
)

df["case_time"] = pd.to_datetime(df["case_time"], format="%H:%M:%S").dt.time
df["case_attention_time"] = pd.to_datetime(
    df["case_attention_time"], format="%H:%M:%S"
).dt.time

### Carga de dimensiones


Primero insertamos los valores únicos en cada tabla de dimensión y construimos
un diccionario de mapeo valor a id para usar en la tabla de hechos.


In [4]:
Markdown(f"Registros a cargar: ${len(df):,}$")

Registros a cargar: $218,591$

#### DimTime


Una fila por cada combinación fecha + hora.


In [5]:
await init_db()

day_name_to_enum = {
    0: "Lunes",
    1: "Martes",
    2: "Miércoles",
    3: "Jueves",
    4: "Viernes",
    5: "Sábado",
    6: "Domingo",
}

turn_map = {
    "Madrugada": "Madrugada",
    "Mañana": "Mañana",
    "Tarde": "Tarde",
    "Noche": "Noche",
}

time_key_to_id: dict[tuple, int] = {}

unique_times_df = df[["case_date", "case_time", "shift"]].drop_duplicates()

async with in_transaction() as connection:
    for _, row in unique_times_df.iterrows():
        date_val = row["case_date"].date()
        time_val = row["case_time"]
        turn_val = row["shift"]
        dt = datetime.combine(date_val, time_val)
        dow = day_name_to_enum[dt.weekday()]

        obj = await DimTime.create(
            date=date_val,
            time=time_val,
            year=date_val.year,
            month=date_val.month,
            day=date_val.day,
            day_of_week=dow,
            turn=turn_val,
            using_db=connection,
        )
        key = (date_val, time_val, turn_val)
        time_key_to_id[key] = obj.pk

await close_db()

Markdown(f"`DimTime`: ${len(time_key_to_id):,}$ registros insertados")

`DimTime`: $95,526$ registros insertados

#### DimLocation


Una fila por cada combinación zona + sector + base.


In [6]:
await init_db()

loc_key_to_id: dict[tuple, int] = {}

unique_locs = df[["zone", "sector", "decentralized_base"]].drop_duplicates()

for _, row in unique_locs.iterrows():
    obj = await DimLocation.create(
        zone_number=int(row["zone"]),
        sector_name=row["sector"],
        decentralized_base=row["decentralized_base"],
    )
    key = (int(row["zone"]), row["sector"], row["decentralized_base"])
    loc_key_to_id[key] = obj.pk

await close_db()

Markdown(f"`DimLocation`: ${len(loc_key_to_id):,}$ registros insertados")

`DimLocation`: $168$ registros insertados

#### DimCaseType


Una fila por cada categoría de caso.


In [7]:
await init_db()

case_key_to_id: dict[str, int] = {}

for category in df["category"].unique():
    obj = await DimCaseType.create(category=category)
    case_key_to_id[category] = obj.pk

Markdown(f"`DimCaseType`: ${len(case_key_to_id):,}$ registros insertados")

`DimCaseType`: $10$ registros insertados

#### DimOrigin


Una fila por cada origen del reporte.


In [8]:
await init_db()

origin_key_to_id: dict[str, int] = {}

for origin in df["origin"].unique():
    obj = await DimOrigin.create(category=origin)
    origin_key_to_id[origin] = obj.pk

Markdown(f"`DimOrigin`: ${len(origin_key_to_id):,}$ registros insertados")

`DimOrigin`: $8$ registros insertados

### Carga de la tabla de hechos

Insertamos `FactIncident` usando los mapeos de las dimensiones. Se procesa en
lotes para manejar eficientemente los $\approx 218 \ 000$ registros.


In [9]:
await init_db()

BATCH_SIZE = 10000

fact_objects = []
total = len(df)

for i, row in df.iterrows():
    if not isinstance(i, int):
        msg = f"Índice no entero encontrado: {i} (tipo {type(i)})"

        raise TypeError(msg)

    time_id = time_key_to_id[
        (row["case_date"].date(), row["case_time"], row["shift"])
    ]
    loc_id = loc_key_to_id[
        (int(row["zone"]), row["sector"], row["decentralized_base"])
    ]
    case_id = case_key_to_id[row["category"]]
    origin_id = origin_key_to_id[row["origin"]]

    response_min = None

    if pd.notna(row["case_time"]) and pd.notna(row["case_attention_time"]):
        t1 = datetime.combine(row["case_date"].date(), row["case_time"])
        t2 = datetime.combine(
            row["case_attention"].date(),
            row["case_attention_time"],
        )
        diff = (t2 - t1).total_seconds() / 60
        response_min = int(diff) if diff >= 0 else None

    fact_objects.append(
        FactIncident(
            case_number=int(row["n_caso"]),
            time_id=time_id,
            location_id=loc_id,
            case_type_id=case_id,
            origin_id=origin_id,
            latitude=Decimal(str(row["latitude"])),
            longitude=Decimal(str(row["longitude"])),
            response_time_min=response_min,
            attention_date=row["case_attention"].date()
            if pd.notna(row["case_attention"])
            else None,
        )
    )

    if len(fact_objects) >= BATCH_SIZE:
        await FactIncident.bulk_create(fact_objects)

        fact_objects = []

if fact_objects:
    await FactIncident.bulk_create(fact_objects)

count = await FactIncident.all().count()

Markdown(f"`FactIncident`: ${count:,}$ registros insertados")

`FactIncident`: $218,591$ registros insertados